# 🛡️ EGSPEC CyberGuard 2026 — Notebook pédagogique
## Détection d'intrusions réseau (épreuve rejouée hors compétition)

**Objectif pédagogique** : reproduire l'intégralité d'une épreuve de hackathon Kaggle de bout en bout —
exploration, découverte du signal, feature engineering, validation croisée, comparaison de modèles,
optimisation du seuil de décision et génération de la soumission.

> ⚠️ **Découverte clé** : la description officielle annonçait 7 classes d'attaques, mais les données
> réelles contiennent une cible **binaire** `Benign` / `Malicious`. Ce notebook travaille sur les
> données réelles. Voir `rapport.md` pour l'analyse complète.

**Résultats obtenus :**
- Macro F1 OOF (XGBoost, seuil 0,5) = **0.9934**
- Macro F1 OOF (XGBoost, **seuil optimisé 0.214**) = **0.9934**
- Les graphiques (`figures/`) ont été produits avec **MATLAB R2024a** en mode batch.


In [ ]:
# %% [1] Chargement des donnees
import pandas as pd
import numpy as np

DATA = r"C:\A\kaggle-egspec-cyberguard-2026\data"
train = pd.read_csv(DATA + r"\TRAIN_DATA.csv")
test = pd.read_csv(DATA + r"\TEST_DATA.csv")

print("train :", train.shape, "| test :", test.shape)
train.head(3)


## 1. Exploration des données (EDA)

Questions auxquelles l'EDA doit répondre avant tout modélisation :
1. Quelle est la cible ? Combien de classes ? Quel déséquilibre ?
2. Y a-t-il des valeurs manquantes, des constantes, des doublons ?
3. Quelles variables portent du signal ?

In [ ]:
# %% [2] EDA : cible, valeurs manquantes, types
print(train["Attack_Label"].value_counts())
print()
print("Pourcentages :")
print((train["Attack_Label"].value_counts(normalize=True) * 100).round(3))
print()
print("Valeurs manquantes :", int(train.isna().sum().sum()))
print("Colonnes non numeriques :", [c for c in train.columns if train[c].dtype == object])


In [ ]:
# %% [3] Correlations avec la cible (encodee 0/1)
y = (train["Attack_Label"] == "Malicious").astype(int)
corr = train.select_dtypes("number").corrwith(y).abs().sort_values(ascending=False)
print(corr.head(10).round(4).to_string())
print()
print("=> Le signal est concentre sur 2 variables :")
print("   Flow_Packets_Per_Sec et SYN_Flag_Count. Tout le reste est du bruit decoratif.")


### 🖼️ Visualisations (générées par MATLAB)

Voir `figures/fig1_classes_et_signal.png`, `figures/fig2_features_cles.png`,
`figures/fig3_seuil_optimal.png`, `figures/fig9_dashboard.png`.

**Loi du générateur découverte dans le train :**

| Zone | P(Malicious) |
|---|---|
| `SYN_Flag_Count ≤ 2` | 0 % |
| `SYN_Flag_Count ≥ 5` | 100 % |
| `SYN ∈ {3,4}` & `FPPS > 5000` | ≈ 26 % |

*Graphique MATLAB correspondant : `fig2_features_cles.png` (nuage bivarié).*


In [ ]:
# %% [4] Verification de la loi du generateur
for syn in [0, 1, 2, 3, 4, 5, 6]:
    m = train.SYN_Flag_Count == syn
    if m.sum():
        print(f"SYN={syn} : n={m.sum():6d}  P(malicious)={y[m].mean():.4f}")


## 2. Cadre de validation

**StratifiedKFold(5), seed 42** — le même cadre pour tous les modèles : chaque ligne est
prédite exactement une fois par un modèle qui ne l'a jamais vue (prédictions *out-of-fold*).
Le Macro F1 OOF est notre estimation honnête de la performance.

⚠️ Ne JAMAIS tuner sur le leaderboard public : ici nous n'avons même pas accès aux labels
du test — le holdout local est le seul juge.

In [ ]:
# %% [5] Cadre de validation commun
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Chaque modele sera entraine 5 fois ; les predictions out-of-fold (OOF)
# sont concatenees pour obtenir un score global sans fuite.
print("Folds :", skf.n_splits)


## 3. Feature engineering

Ratios et interactions "domaine réseau" : octets par paquet, ratios fwd/bwd,
proportions de flags, ratio actif/idle… Sur ce dataset synthétique ils n'améliorent que
marginalement (le signal est déjà dans 2 colonnes brutes), mais la démarche est celle
attendue dans une épreuve réelle.

In [ ]:
# %% [6] Feature engineering reutilisable
def engineer(df):
    d = df.copy()
    d["bytes_per_packet"] = (d.Total_Fwd_Bytes + d.Total_Bwd_Bytes) / (d.Total_Fwd_Packets + d.Total_Bwd_Packets + 1)
    d["fwd_bwd_bytes_ratio"] = d.Total_Fwd_Bytes / (d.Total_Bwd_Bytes + 1)
    d["fwd_bwd_pkts_ratio"] = d.Total_Fwd_Packets / (d.Total_Bwd_Packets + 1)
    flag_cols = ["SYN_Flag_Count", "ACK_Flag_Count", "RST_Flag_Count",
                 "PSH_Flag_Count", "FIN_Flag_Count", "URG_Flag_Count"]
    d["flags_total"] = d[flag_cols].sum(axis=1)
    d["syn_ratio"] = d.SYN_Flag_Count / (d.flags_total + 1)
    d["rst_ratio"] = d.RST_Flag_Count / (d.flags_total + 1)
    d["active_idle_ratio"] = d.Active_Mean / (d.Idle_Mean + 1)
    d["iat_std_ratio"] = d.Flow_IAT_Std / (d.Flow_IAT_Mean + 1)
    d = pd.get_dummies(d, columns=["Protocol"], prefix="proto")
    return d
print("OK — fonction engineer() definie")


## 4. Comparaison de modèles

Quatre familles testées **dans le même cadre** (mêmes folds, mêmes features) :
régression logistique (linéaire), forêt aléatoire (bagging), XGBoost (boosting),
perceptron multicouche (réseau de neurones). Plus la règle experte en référence.

**Résultats OOF mesurés (seuil 0,5) :**

 | Modele | Macro F1 OOF (seuil 0,5) | Ecart-type folds |
|---|---|---|
| LogisticRegression | 0.9854 | ± 0.0007 |
| MLP | 0.9839 | ± 0.0011 |
| RandomForest | 0.9934 | ± 0.0006 |
| XGBoost | 0.9934 | ± 0.0006 |

Avec **seuil optimisé** par modèle :
- **LogisticRegression** → Macro F1 = 0.9854
- **RandomForest** → Macro F1 = 0.9934
- **XGBoost** → Macro F1 = 0.9934
- **MLP** → Macro F1 = 0.9839
- **XGBoost (seuil optimise 0.214)** → Macro F1 = 0.9934
- **LogisticRegression (seuil optimise 0.617)** → Macro F1 = 0.9885
- **RandomForest (seuil optimise 0.348)** → Macro F1 = 0.9934
- **XGBoost (seuil optimise 0.214)** → Macro F1 = 0.9934
- **MLP (seuil optimise 0.585)** → Macro F1 = 0.9843

*Graphiques MATLAB : `fig4_comparaison_modeles.png`, `fig6_matrices_confusion.png`.*


In [ ]:
# %% [7] Reproduire la comparaison (exemple : RandomForest seul, ~2 min/fold)
# Le script complet (4 modeles x 5 folds) est dans compare_models.py — il a deja tourne.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

Xe = engineer(train.drop(columns=["Attack_Label"]))
m = RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample",
                           random_state=42, n_jobs=-1)
skf2 = StratifiedKFold(5, shuffle=True, random_state=42)
scores = []
for tr_i, va_i in skf2.split(Xe, y):
    m.fit(Xe.iloc[tr_i], y.iloc[tr_i])
    scores.append(f1_score(y.iloc[va_i], m.predict(Xe.iloc[va_i]), average="macro"))
print("RandomForest macroF1 par fold :", [round(s, 4) for s in scores])


## 5. Optimisation du seuil de décision

Le seuil 0,5 n'est **pas** optimal pour le Macro F1 : la classe majoritaire étant bénigne,
baisser le seuil augmente le rappel de la classe minoritaire… jusqu'à ce que les faux positifs
grignotent son F1. On balaie 395 seuils sur les prédictions **OOF** (jamais vues à l'entraînement)
et on garde l'argmax.

**Résultat : seuil optimal = 0.214 → Macro F1 = 0.9934** (vs 0.9934 à 0,5).

*Graphique MATLAB : `fig3_seuil_optimal.png` — la courbe seuil → Macro F1 avec l'optimum marqué.*


In [ ]:
# %% [8] Balayage du seuil sur les predictions OOF
# (les OOF sont deja calculees par compare_models.py ; ici on demontre la methode
#  avec un modele unique train/test pour rester rapide)
from sklearn.model_selection import train_test_split

Xa, Xb, ya, yb = train_test_split(Xe, y, test_size=0.2, random_state=42, stratify=y)
from xgboost import XGBClassifier
m2 = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                   scale_pos_weight=(ya == 0).sum() / (ya == 1).sum(),
                   eval_metric="logloss", random_state=42, n_jobs=-1, tree_method="hist")
m2.fit(Xa, ya)
pb = m2.predict_proba(Xb)[:, 1]

ths = np.linspace(0.01, 0.99, 395)
curve = [f1_score(yb, (pb > t).astype(int), average="macro") for t in ths]
best = int(np.argmax(curve))
print(f"Seuil optimal (sur ce holdout) = {ths[best]:.3f} -> macroF1 = {curve[best]:.4f}")


## 6. Le piège du test : un test "sans attaques"

Diagnostic crucial de cette épreuve :
- `SYN_Flag_Count` du test ne prend **que** les valeurs 0–4 (jamais ≥ 5)
- `Flow_Packets_Per_Sec` du test culmine à ≈ 10 000 (plage bénigne)
- Le modèle validé à 0,9934 en OOF prédit **0 malicieux sur 25 000** lignes du test
  (probabilité max = 0,15)

**Conclusion** : le test a été généré depuis la distribution bénigne uniquement.
La soumission optimale était donc `Benign` partout — le genre de piège qu'un contrôle
systématique des distributions train/test permet d'attraper *avant* de soumettre.

In [ ]:
# %% [9] Diagnostic train vs test sur les features cles
print("SYN_Flag_Count train :", sorted(train.SYN_Flag_Count.unique()))
print("SYN_Flag_Count test  :", sorted(test.SYN_Flag_Count.unique()))
print("FPPS test max :", round(test.Flow_Packets_Per_Sec.max(), 1),
      "| FPPS train max :", round(train.Flow_Packets_Per_Sec.max(), 1))


## 7. Génération de la soumission

Format attendu : 2 colonnes `Flow_ID`, `Attack_Label` (valeurs réelles du dataset :
`Benign` / `Malicious`).

In [ ]:
# %% [10] Soumission (deja generee par compare_models.py avec le seuil optimal)
import os
sub = pd.read_csv(r"C:\A\kaggle-egspec-cyberguard-2026\submission.csv")
print(sub.shape)
print(sub["Attack_Label"].value_counts())
sub.head()


## 8. Courbe d'apprentissage et leçons

Mesures (XGBoost, holdout 20 %) : 25 arbres → 0.9939, 50 arbres → 0.9939, 100 arbres → 0.9939, 200 arbres → 0.9939, 400 arbres → 0.9939, 800 arbres → 0.9939.
Au-delà de ~300 arbres, le gain devient marginal : inutile de brûler du CPU.

### Leçons de l'épreuve
1. **Valider l'énoncé contre les données** (10 min d'EDA ont invalidé 3 affirmations de l'annonce).
2. **Chercher la frontière de décision avant de complexifier** : 2 features donnaient déjà 0,938.
3. **Le holdout local est le seul juge fiable** — surtout quand le test diverge du train.
4. **Vérifier la distribution des features prédictives du test** avant de finaliser.
5. **Optimiser le seuil sur l'OOF**, jamais sur le leaderboard public.

*Courbes MATLAB : `fig5_roc_pr.png`, `fig7_importance_features.png`, `fig8_courbe_apprentissage.png`.*
